# Milestone 4: Dimensionality Reduction and Second ModelThis notebook covers the second model for the EB-NeRD click prediction project:1. **Train Your Second Model using Dimensionality Reduction**: reduce the feature space with PCA, then train a GBT classifier on the reduced features2. **Evaluate Your Model**: report AUC-ROC, AUC-PR, accuracy, and F1 on the train, test, and validation splits3. **Fitting Analysis**: compare train, test, and validation performance and place the model on the fitting spectrum relative to the MS3 baselines4. **Update README.md**: document the project, methods, results, and links to the notebooks in the repository README5. **Conclusion**: summarize what the first and second models show, and what could improve the system next6. **Predictions Analysis**: inspect true positives, true negatives, false positives, and false negatives on the test set**Dataset:** EB-NeRD `ebnerd_large` on SDSC Expanse, about 38M impression logs downsampled to about 20M training candidate rows after negative sampling.  **Infrastructure:** SDSC Expanse, 16 CPU / 128 GB node, PySpark `local[15]`.

## Section 1: Train Your Second Model using Dimensionality Reduction

### Setup & SparkSession

#### Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
from pyspark.ml import Pipeline
from pyspark.ml.feature import PCA, VectorAssembler, StringIndexer, OneHotEncoder, Imputer, StandardScaler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import getpass

DATA_ROOT       = os.path.expanduser("~/ebnerd_data")
TRAIN_VAL_DIR   = os.path.join(DATA_ROOT, "ebnerd_large")
ARTIFACTS_DIR   = os.path.join(DATA_ROOT, "artifacts")
OUTPUT_DIR      = os.path.join(DATA_ROOT, "ms3_output")   # reuse MS3 pipeline artifacts
MS4_OUTPUT_DIR  = os.path.join(DATA_ROOT, "ms4_output")

SCRATCH_ROOT   = f"/expanse/lustre/scratch/{getpass.getuser()}/temp_project"
CHECKPOINT_DIR = os.path.join(SCRATCH_ROOT, "spark_checkpoints_ms4")
LOCAL_DIR      = os.path.join(SCRATCH_ROOT, "spark_local")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR,      exist_ok=True)
os.makedirs(MS4_OUTPUT_DIR, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("EB-NeRD MS4 Dimensionality Reduction & Modeling")
    .master("local[15]")
    .config("spark.driver.memory", "96g")
    .config("spark.driver.maxResultSize", "16g")
    .config("spark.sql.shuffle.partitions", "800")
    .config("spark.local.dir", LOCAL_DIR)
    .config("spark.sql.parquet.enableVectorizedReader", "true")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")
    .config("spark.checkpoint.compress", "true")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.3")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)

print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")
print(f"Executors     : {spark.sparkContext.defaultParallelism}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

### 1.2 Rebuild Preprocessed Feature SetsThis section rebuilds the preprocessed train, test, and validation splits by rerunning the MS3 feature engineering steps and reloading the saved preprocessing pipeline model. That keeps the MS4 notebook self-contained without storing hundreds of GB of intermediate parquet files. The pipeline is fit only on the training split and then applied to test and validation so there is no leakage.

In [ ]:
import glob as globlib
from pyspark.ml import PipelineModel

# ── Load raw tables ──────────────────────────────────────────────────────────
df_behaviors_train = spark.read.parquet(os.path.join(TRAIN_VAL_DIR, "train",      "behaviors.parquet"))
df_behaviors_val   = spark.read.parquet(os.path.join(TRAIN_VAL_DIR, "validation", "behaviors.parquet"))
df_history_train   = spark.read.parquet(os.path.join(TRAIN_VAL_DIR, "train",      "history.parquet"))
df_history_val     = spark.read.parquet(os.path.join(TRAIN_VAL_DIR, "validation", "history.parquet"))
df_articles        = spark.read.parquet(os.path.join(TRAIN_VAL_DIR, "articles.parquet"))

CONTRASTIVE_DIR = os.path.join(ARTIFACTS_DIR, "Ekstra_Bladet_contrastive_vector")
contrastive_candidates = globlib.glob(os.path.join(CONTRASTIVE_DIR, "**", "*.parquet"), recursive=True)
df_contrastive = spark.read.parquet(contrastive_candidates[0] if contrastive_candidates else CONTRASTIVE_DIR)

df_history = df_history_train.unionByName(df_history_val, allowMissingColumns=True)

# ── Reproduce MS3 helper functions ───────────────────────────────────────────
def explode_impressions(df_behaviors, split_name):
    return (
        df_behaviors
        .select(
            "impression_id", "user_id", "impression_time", "read_time",
            "scroll_percentage", "device_type", "is_sso_user", "is_subscriber",
            "gender", "age", "article_ids_clicked",
            F.explode("article_ids_inview").alias("candidate_article_id")
        )
        .withColumn(
            "label",
            F.when(
                F.array_contains(F.col("article_ids_clicked"), F.col("candidate_article_id")),
                F.lit(1.0)
            ).otherwise(F.lit(0.0))
        )
        .drop("article_ids_clicked")
        .withColumn("split", F.lit(split_name))
    )

from pyspark.sql import Window
NPRATIO = 4

def negative_downsample(df_exploded, npratio, seed=42):
    w_neg = Window.partitionBy("impression_id").orderBy(F.rand(seed=seed))
    df_pos = df_exploded.filter(F.col("label") == 1.0)
    pos_per_imp = df_pos.groupBy("impression_id").agg(F.count("*").alias("n_pos"))
    df_neg_filtered = (
        df_exploded.filter(F.col("label") == 0.0)
        .withColumn("neg_rank", F.row_number().over(w_neg))
        .join(pos_per_imp, "impression_id", "left")
        .fillna({"n_pos": 0})
        .filter(F.col("neg_rank") <= F.col("n_pos") * npratio)
        .drop("neg_rank", "n_pos")
    )
    return df_pos.unionByName(df_neg_filtered)

def fill_behavior_nulls(df):
    return (
        df
        .fillna({"scroll_percentage": 0.0, "read_time": 0.0})
        .fillna({"gender": -1, "age": -1})
        .fillna({"is_sso_user": False, "is_subscriber": False})
        .drop("postcode", "article_ids_clicked", "split")
        if "postcode" in df.columns
        else df.fillna({"scroll_percentage": 0.0, "read_time": 0.0})
                .fillna({"gender": -1, "age": -1})
                .fillna({"is_sso_user": False, "is_subscriber": False})
    )

print("Raw tables loaded.")

In [ ]:
# ── Article features ─────────────────────────────────────────────────────────
df_art_features = (
    df_articles
    .select(
        "article_id", "published_time", "premium", "category_str",
        "sentiment_score", "sentiment_label",
        "total_inviews", "total_pageviews", "total_read_time",
        F.size(F.split(F.coalesce(F.col("title"),    F.lit("")), r"\s+")).alias("title_word_count"),
        F.size(F.split(F.coalesce(F.col("subtitle"), F.lit("")), r"\s+")).alias("subtitle_word_count"),
        F.size(F.split(F.coalesce(F.col("body"),     F.lit("")), r"\s+")).alias("body_word_count"),
        F.size(F.coalesce(F.col("topics"),       F.array())).alias("n_topics"),
        F.size(F.coalesce(F.col("ner_clusters"), F.array())).alias("n_entities"),
        F.size(F.coalesce(F.col("subcategory"),  F.array())).alias("n_subcategories"),
    )
    .fillna({
        "total_inviews": 0, "total_pageviews": 0, "total_read_time": 0.0,
        "sentiment_score": 0.5, "sentiment_label": "Neutral",
        "premium": False,       "category_str": "unknown",
    })
)

# ── User history features ─────────────────────────────────────────────────────
df_user_hist_features = (
    df_history
    .select(
        "user_id",
        F.size("article_id_fixed").alias("user_history_length"),
        F.expr("aggregate(read_time_fixed, 0.0D, (acc, x) -> acc + coalesce(x, 0.0D))").alias("_utr"),
    )
    .withColumn(
        "user_avg_read_time_hist",
        F.when(F.col("user_history_length") > 0,
               F.col("_utr") / F.col("user_history_length")).otherwise(F.lit(0.0))
    )
    .drop("_utr")
)

# ── Explode & downsample ──────────────────────────────────────────────────────
df_candidates_train = explode_impressions(df_behaviors_train, "train")
df_candidates_val   = explode_impressions(df_behaviors_val,   "val")

df_train_sampled = negative_downsample(df_candidates_train, NPRATIO).cache()
df_val_sampled   = negative_downsample(df_candidates_val,   NPRATIO).cache()

# ── Temporal split ────────────────────────────────────────────────────────────
df_train_sampled.createOrReplaceTempView("train_sampled")
cutoff_ts = spark.sql("""
    SELECT percentile_approx(unix_timestamp(impression_time), 0.80) AS cutoff
    FROM train_sampled
""").collect()[0]["cutoff"]

from datetime import datetime, timezone
print(f"Train/test cutoff: {datetime.fromtimestamp(cutoff_ts, tz=timezone.utc)}")

df_train_split = df_train_sampled.filter(F.unix_timestamp("impression_time") <  cutoff_ts)
df_test_split  = df_train_sampled.filter(F.unix_timestamp("impression_time") >= cutoff_ts)
df_val_split   = df_val_sampled

# ── Rolling 24h CTR ───────────────────────────────────────────────────────────
df_hourly = (
    df_candidates_train
    .withColumn("hour_bucket", (F.floor(F.unix_timestamp("impression_time") / 3600) * 3600).cast("long"))
    .groupBy("candidate_article_id", "hour_bucket")
    .agg(F.sum("label").alias("hourly_clicks"), F.count("*").alias("hourly_impressions"))
    .withColumnRenamed("candidate_article_id", "article_id")
)
w_rolling = (
    Window.partitionBy("article_id").orderBy("hour_bucket")
    .rangeBetween(-24 * 3600, -1)
)
df_rolling_ctr = (
    df_hourly
    .withColumn("rolling_clicks_24h",      F.sum("hourly_clicks").over(w_rolling))
    .withColumn("rolling_impressions_24h", F.sum("hourly_impressions").over(w_rolling))
    .withColumn("rolling_ctr_24h",
        F.when(F.col("rolling_impressions_24h") > 0,
               F.col("rolling_clicks_24h") / F.col("rolling_impressions_24h")).otherwise(F.lit(0.0)))
    .withColumn("rolling_popularity_24h", F.col("rolling_impressions_24h").cast("double"))
    .select("article_id", "hour_bucket", "rolling_ctr_24h", "rolling_popularity_24h")
    .fillna({"rolling_ctr_24h": 0.0, "rolling_popularity_24h": 0.0})
)

print("Feature tables ready.")

In [ ]:
CAT_COLS = ["category_str", "sentiment_label"]
NUM_COLS = [
    "read_time", "scroll_percentage", "impression_hour", "impression_weekday",
    "article_age_hours", "log_article_age_hours", "log_total_inviews",
    "log_total_pageviews", "total_read_time", "sentiment_score",
    "title_word_count", "subtitle_word_count", "body_word_count",
    "n_topics", "n_entities", "n_subcategories",
    "rolling_ctr_24h", "rolling_popularity_24h",
    "user_history_length", "user_avg_read_time_hist",
]
BIN_COLS = ["premium_flag", "is_cold_article", "is_subscriber_flag", "is_sso_flag"]
ORD_COLS = ["device_type", "gender", "age"]

def join_all_features(df_split, df_art_feats, df_user_hist, df_rolling):
    df = df_split.withColumn(
        "hour_bucket", (F.floor(F.unix_timestamp("impression_time") / 3600) * 3600).cast("long")
    )
    df = df.join(df_art_feats.withColumnRenamed("article_id", "candidate_article_id"),
                 on="candidate_article_id", how="left")
    df = df.join(df_user_hist, on="user_id", how="left")
    df = df.join(df_rolling.withColumnRenamed("article_id", "candidate_article_id"),
                 on=["candidate_article_id", "hour_bucket"], how="left")
    df = (
        df
        .withColumn("impression_hour",    F.hour("impression_time").cast("double"))
        .withColumn("impression_weekday", F.dayofweek("impression_time").cast("double"))
        .withColumn(
            "article_age_hours",
            F.when(F.col("published_time").isNotNull(),
                   (F.unix_timestamp("impression_time") - F.unix_timestamp("published_time")) / 3600.0
            ).otherwise(F.lit(720.0))
        )
        .withColumn("log_article_age_hours", F.log1p(F.col("article_age_hours")))
        .withColumn(
            "is_cold_article",
            F.when((F.col("article_age_hours") <= 24.0) | (F.col("total_inviews") == 0),
                   F.lit(1.0)).otherwise(F.lit(0.0))
        )
        .withColumn("log_total_inviews",   F.log1p(F.col("total_inviews").cast("double")))
        .withColumn("log_total_pageviews", F.log1p(F.col("total_pageviews").cast("double")))
        .withColumn("premium_flag",       F.col("premium").cast("double"))
        .withColumn("is_subscriber_flag", F.col("is_subscriber").cast("double"))
        .withColumn("is_sso_flag",        F.col("is_sso_user").cast("double"))
    )
    return df.fillna({
        "user_history_length": 0, "user_avg_read_time_hist": 0.0,
        "rolling_ctr_24h": 0.0,  "rolling_popularity_24h": 0.0,
        "sentiment_score": 0.5,  "n_topics": 0, "n_entities": 0,
        "n_subcategories": 0,    "title_word_count": 0,
        "subtitle_word_count": 0, "body_word_count": 0,
        "total_read_time": 0.0,  "category_str": "unknown",
        "sentiment_label": "Neutral", "device_type": 0,
    })

def cast_ordinal_cols(df):
    for col in ORD_COLS:
        df = df.withColumn(f"{col}_dbl", F.col(col).cast("double"))
    return df

df_train_feat = cast_ordinal_cols(join_all_features(df_train_split, df_art_features, df_user_hist_features, df_rolling_ctr))
df_test_feat  = cast_ordinal_cols(join_all_features(df_test_split,  df_art_features, df_user_hist_features, df_rolling_ctr))
df_val_feat   = cast_ordinal_cols(join_all_features(df_val_split,   df_art_features, df_user_hist_features, df_rolling_ctr))

# ── Load saved MS3 preprocessing pipeline ────────────────────────────────────
preprocessing_pipeline_model = PipelineModel.load(os.path.join(OUTPUT_DIR, "preprocessing_pipeline"))
print("Loaded MS3 preprocessing pipeline.")

df_train_feat = df_train_feat.cache()
df_test_feat  = df_test_feat.cache()
df_val_feat   = df_val_feat.cache()

df_train_prep = preprocessing_pipeline_model.transform(df_train_feat)
df_test_prep  = preprocessing_pipeline_model.transform(df_test_feat)
df_val_prep   = preprocessing_pipeline_model.transform(df_val_feat)

keep_cols = ["impression_id", "user_id", "candidate_article_id", "label", "features", "is_cold_article"]
df_train_prep = df_train_prep.select(keep_cols).cache()
df_test_prep  = df_test_prep.select(keep_cols).cache()
df_val_prep   = df_val_prep.select(keep_cols).cache()

df_train_prep = df_train_prep.checkpoint()
df_test_prep  = df_test_prep.checkpoint()
df_val_prep   = df_val_prep.checkpoint()

df_train_feat.unpersist(); df_test_feat.unpersist(); df_val_feat.unpersist()

sample_row = df_train_prep.select("features").first()
n_features = len(sample_row["features"])
print(f"Feature vector dimension (full): {n_features}")
print(f"Train rows : {df_train_prep.count():,}")
print(f"Test rows  : {df_test_prep.count():,}")
print(f"Val rows   : {df_val_prep.count():,}")

### Dimensionality Reduction with PCA#### Why PCA for this dataset?The preprocessed feature vector has about 30 dimensions after combining numeric features, one-hot encoded categoricals, binary flags, and ordinal fields. That is still manageable for tree models, but PCA is useful here for a few practical reasons:1. **Decorrelation**: features like `log_total_inviews`, `log_total_pageviews`, `total_read_time`, and `rolling_popularity_24h` all describe article engagement and are strongly correlated. PCA compresses that overlap into orthogonal components.2. **Visualization**: the first two principal components give a 2D view of how clicked and non-clicked examples separate in feature space.3. **Compression**: we can test whether a smaller set of components, capturing about 90 to 95% of the variance, preserves most of the predictive signal.4. **Cold-start insight**: if cold articles cluster in a distinct part of PCA space, that supports the idea that cold-start is a feature-separability issue rather than just a model-capacity issue.PCA is fit on the **training set only**, and the fitted model is then applied to all three splits. In Spark, `pyspark.ml.feature.PCA` computes the covariance information in a distributed way and then performs the matrix decomposition on the driver.

### 2.2 Fit PCA at Full Rank (k = n_features)We first fit PCA at full rank to recover all eigenvalues, then plot the explained variance curve to choose the right k.

In [ ]:
# PCA full rank: fit on training data only to get all eigenvaluespca_full = PCA(k=n_features, inputCol="features", outputCol="pca_full_features")pca_full_model = pca_full.fit(df_train_prep)print("Full-rank PCA fitted.")# Explained varianceexplained_var  = np.array(pca_full_model.explainedVariance.toArray())cumulative_var = np.cumsum(explained_var)print(f"\nTop-10 eigenvalues (explained variance ratio):")for i, v in enumerate(explained_var[:10]):    print(f"  PC{i+1:02d}: {v:.4f}  (cumulative: {cumulative_var[i]:.4f})")k_90 = int(np.searchsorted(cumulative_var, 0.90)) + 1k_95 = int(np.searchsorted(cumulative_var, 0.95)) + 1k_99 = int(np.searchsorted(cumulative_var, 0.99)) + 1print(f"\nComponents needed to reach:")print(f"  90% variance: k = {k_90}")print(f"  95% variance: k = {k_95}")print(f"  99% variance: k = {k_99}")

### 2.3 Explained Variance Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))ks = np.arange(1, len(explained_var) + 1)# Individual explained variance (scree plot)ax = axes[0]ax.bar(ks, explained_var, color="steelblue", edgecolor="white")ax.set_title("Scree Plot: Individual Explained Variance")ax.set_xlabel("Principal Component")ax.set_ylabel("Explained Variance Ratio")ax.set_xlim(0.5, len(explained_var) + 0.5)ax.grid(axis="y", alpha=0.3)# Cumulative explained varianceax = axes[1]ax.plot(ks, cumulative_var, marker="o", markersize=4, color="steelblue")for thresh, k_val, color in [(0.90, k_90, "orange"), (0.95, k_95, "red"), (0.99, k_99, "purple")]:    ax.axhline(thresh, linestyle="--", color=color, alpha=0.7,               label=f"{int(thresh*100)}% @ k={k_val}")    ax.axvline(k_val,  linestyle=":",  color=color, alpha=0.5)ax.set_title("Cumulative Explained Variance vs. k")ax.set_xlabel("Number of Components (k)")ax.set_ylabel("Cumulative Explained Variance")ax.set_ylim(0, 1.05)ax.legend()ax.grid(alpha=0.3)plt.suptitle("PCA Explained Variance: EB-NeRD Feature Space", fontsize=13)plt.tight_layout()plt.savefig(os.path.join(MS4_OUTPUT_DIR, "pca_explained_variance.png"), dpi=150, bbox_inches="tight")plt.show()print(f"Plot saved to {MS4_OUTPUT_DIR}/pca_explained_variance.png")

### 2.4 Component InterpretationTo understand what each principal component is capturing, we inspect the loading vectors for PC1 and PC2. Because the final feature vector mixes one-hot encoded, numeric, and binary inputs, we recover the original feature names from the `ml_attr` metadata attached to the assembled `features` column.

In [ ]:
# Recover feature names from assembled vector metadatametadata = (    df_train_prep.schema["features"].metadata    .get("ml_attr", {}).get("attrs", {}))feat_names_map = {}for attr_type in ["numeric", "binary", "nominal"]:    for attr in metadata.get(attr_type, []):        feat_names_map[attr["idx"]] = attr["name"]n_feat = pca_full_model.pc.numRowsfeat_labels = [feat_names_map.get(i, f"feat_{i}") for i in range(n_feat)]# PC1 and PC2 loadingspc_matrix = np.array(pca_full_model.pc.toArray())  # shape (n_features, k)pc1_loadings = pc_matrix[:, 0]pc2_loadings = pc_matrix[:, 1]def plot_loadings(loadings, pc_label, ax, top_n=15):    sorted_idx = np.argsort(np.abs(loadings))[::-1][:top_n]    names  = [feat_labels[i] for i in sorted_idx][::-1]    values = [loadings[i] for i in sorted_idx][::-1]    colors = ["steelblue" if v >= 0 else "coral" for v in values]    ax.barh(names, values, color=colors, edgecolor="white")    ax.axvline(0, color="black", linewidth=0.8)    ax.set_title(f"{pc_label} Loadings (top {top_n} by |loading|)")    ax.set_xlabel("Loading")    ax.grid(axis="x", alpha=0.3)fig, axes = plt.subplots(1, 2, figsize=(16, 7))plot_loadings(pc1_loadings, "PC1", axes[0])plot_loadings(pc2_loadings, "PC2", axes[1])plt.suptitle("PCA Component Loadings: PC1 and PC2", fontsize=13)plt.tight_layout()plt.savefig(os.path.join(MS4_OUTPUT_DIR, "pca_loadings.png"), dpi=150, bbox_inches="tight")plt.show()print("\nPC1 top 5 features (by absolute loading):")for i in np.argsort(np.abs(pc1_loadings))[::-1][:5]:    print(f"  {feat_labels[i]:<40} loading={pc1_loadings[i]:+.4f}")print("\nPC2 top 5 features (by absolute loading):")for i in np.argsort(np.abs(pc2_loadings))[::-1][:5]:    print(f"  {feat_labels[i]:<40} loading={pc2_loadings[i]:+.4f}")

### 2.5 PCA 2-D Projection (Visualization)

We sample 50,000 rows from the test set and project onto PC1/PC2 to visualize the click vs. no-click decision boundary, with cold articles highlighted separately.

In [ ]:
# Fit 2-component PCA for visualizationpca_2d = PCA(k=2, inputCol="features", outputCol="pca_2d_features")pca_2d_model = pca_2d.fit(df_train_prep)df_test_2d = (    pca_2d_model.transform(df_test_prep)    .select(        F.round(vector_to_array(F.col("pca_2d_features"))[0], 6).alias("pc1"),        F.round(vector_to_array(F.col("pca_2d_features"))[1], 6).alias("pc2"),        "label", "is_cold_article"    )    .sample(fraction=50_000 / df_test_prep.count(), seed=42)    .toPandas())print(f"Projection sample: {len(df_test_2d):,} rows")fig, axes = plt.subplots(1, 2, figsize=(16, 6))# Panel 1: click vs. no-clickax = axes[0]for label_val, label_str, color, alpha in [    (0.0, "No-click", "#4C72B0", 0.15),    (1.0, "Click",    "#DD8452", 0.50),]:    sub = df_test_2d[df_test_2d["label"] == label_val]    ax.scatter(sub["pc1"], sub["pc2"], s=2, alpha=alpha, color=color, label=f"{label_str} (n={len(sub):,})", rasterized=True)ax.set_title("PC1 vs PC2: Click vs. No-Click")ax.set_xlabel(f"PC1 ({100*explained_var[0]:.1f}% variance)")ax.set_ylabel(f"PC2 ({100*explained_var[1]:.1f}% variance)")ax.legend(markerscale=5)ax.grid(alpha=0.2)# Panel 2: cold vs. warmax = axes[1]for cold_val, cold_str, color, alpha in [    (0.0, "Warm article", "#55A868", 0.15),    (1.0, "Cold article", "#C44E52", 0.30),]:    sub = df_test_2d[df_test_2d["is_cold_article"] == cold_val]    ax.scatter(sub["pc1"], sub["pc2"], s=2, alpha=alpha, color=color, label=f"{cold_str} (n={len(sub):,})", rasterized=True)ax.set_title("PC1 vs PC2: Cold vs. Warm Articles")ax.set_xlabel(f"PC1 ({100*explained_var[0]:.1f}% variance)")ax.set_ylabel(f"PC2 ({100*explained_var[1]:.1f}% variance)")ax.legend(markerscale=5)ax.grid(alpha=0.2)plt.suptitle("PCA 2-D Projection: Test Set Sample (n=50k)", fontsize=13)plt.tight_layout()plt.savefig(os.path.join(MS4_OUTPUT_DIR, "pca_2d_projection.png"), dpi=150, bbox_inches="tight")plt.show()

### 2.6 Select k for ModelingBased on the explained variance curve above, we use **k = k_95**, the number of components needed to capture 95% of the variance, for the second model. This gives a reasonable balance between compression and information retention: we shrink the input space, remove low-variance directions, and still keep nearly all of the signal.

In [ ]:
K_MODEL = k_95   # components capturing 95% varianceprint(f"Selected k = {K_MODEL}  (captures {100*cumulative_var[K_MODEL-1]:.2f}% of variance)")print(f"Compression ratio: {n_features} to {K_MODEL}  ({100*(1 - K_MODEL/n_features):.1f}% reduction)")# Fit the k=K_MODEL PCA on training datapca_model = PCA(k=K_MODEL, inputCol="features", outputCol="pca_features")pca_fitted = pca_model.fit(df_train_prep)print(f"\nPCA model fitted: {K_MODEL} components")# Transform all splitsdf_train_pca = pca_fitted.transform(df_train_prep).select(    "impression_id", "user_id", "candidate_article_id", "label", "pca_features", "is_cold_article").cache()df_test_pca = pca_fitted.transform(df_test_prep).select(    "impression_id", "user_id", "candidate_article_id", "label", "pca_features", "is_cold_article").cache()df_val_pca = pca_fitted.transform(df_val_prep).select(    "impression_id", "user_id", "candidate_article_id", "label", "pca_features", "is_cold_article").cache()df_train_pca = df_train_pca.checkpoint()df_test_pca  = df_test_pca.checkpoint()df_val_pca   = df_val_pca.checkpoint()pca_fitted.write().overwrite().save(os.path.join(MS4_OUTPUT_DIR, "pca_model"))print("PCA model saved.")print(f"\nTrain PCA rows : {df_train_pca.count():,}")print(f"Test PCA rows  : {df_test_pca.count():,}")print(f"Val PCA rows   : {df_val_pca.count():,}")

### GBT Classifier on PCA Features

#### Model Design

We train a GBT classifier on the PCA-reduced features (`pca_features`, k components). The hyperparameters mirror the best MS3 model (GBT tuned) so that the only variable between the two experiments is the input feature space (full ~30-dim vs. PCA k-dim). Any difference in AUC directly reflects the information loss or noise reduction from PCA.

| | MS3 GBT Tuned (baseline) | MS4 GBT on PCA |
|---|---|---|
| Input features | Full ~30-dim standard-scaled vector | PCA k-dim projected vector |
| `maxIter` | 50 | 50 |
| `maxDepth` | 7 | 7 |
| `stepSize` | 0.05 | 0.05 |
| `subsamplingRate` | 0.7 | 0.7 |
| `minInstancesPerNode` | 5 | 5 |
| `featureSubsetStrategy` | sqrt | sqrt |

In [ ]:
gbt_pca = GBTClassifier(
    labelCol="label",
    featuresCol="pca_features",
    maxIter=50,
    maxDepth=7,
    stepSize=0.05,
    subsamplingRate=0.7,
    featureSubsetStrategy="sqrt",
    minInstancesPerNode=5,
    seed=42,
    maxBins=64,
)

print(f"Training GBT on PCA-{K_MODEL} features (maxIter=50, maxDepth=7, stepSize=0.05)...")
gbt_pca_fitted = gbt_pca.fit(df_train_pca)
print(f"Training complete. Trees: {len(gbt_pca_fitted.trees)}")

## Section 2: Evaluate Your Model

### Performance Metrics on Train/Test/Val Splits

### 2.1 What is being evaluated here?This section covers the evaluation requirements for the PCA path used in Milestone 4:- It compares train, test, and validation performance for the supervised model built on reduced features.- It references the explained variance analysis from Section 1, since PCA quality is part of model evaluation here.- Clustering metrics are not included because this notebook follows the PCA plus supervised-model route rather than clustering on the reduced features.

In [ ]:
auc_evaluator   = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
aucpr_evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR")
acc_evaluator   = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
f1_evaluator    = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

def evaluate_model(fitted_model, splits_dict, model_name):
    results = {}
    for split_name, df_split in splits_dict.items():
        preds = fitted_model.transform(df_split)
        auc   = auc_evaluator.evaluate(preds)
        aucpr = aucpr_evaluator.evaluate(preds)
        acc   = acc_evaluator.evaluate(preds)
        f1    = f1_evaluator.evaluate(preds)
        results[split_name] = {"AUC-ROC": auc, "AUC-PR": aucpr, "Accuracy": acc, "F1": f1}
        print(f"  [{model_name}] {split_name:<6} | AUC-ROC={auc:.4f}  AUC-PR={aucpr:.4f}  Acc={acc:.4f}  F1={f1:.4f}")
    return results

print(f"=== Model 4: GBT on PCA-{K_MODEL} features ===")
pca_splits = {"train": df_train_pca, "test": df_test_pca, "val": df_val_pca}
gbt_pca_results = evaluate_model(gbt_pca_fitted, pca_splits, f"GBT-PCA{K_MODEL}")

gbt_pca_fitted.write().overwrite().save(os.path.join(MS4_OUTPUT_DIR, f"model4_gbt_pca{K_MODEL}"))
print(f"\nModel saved to {MS4_OUTPUT_DIR}/model4_gbt_pca{K_MODEL}")

## Section 3: Fitting Analysis

### 3.1 Full Model Comparison (MS3 + MS4)

In [ ]:
# MS3 results from notebook outputs, hardcoded for the comparison tablems3_results = {    "RF (numTrees=50, maxDepth=8)": {        "train": {"AUC-ROC": 0.7123, "AUC-PR": 0.2621},        "test":  {"AUC-ROC": 0.7164, "AUC-PR": 0.2649},        "val":   {"AUC-ROC": 0.6538, "AUC-PR": 0.2803},    },    "GBT default (maxIter=20, lr=0.1)": {        "train": {"AUC-ROC": 0.7200, "AUC-PR": 0.2698},        "test":  {"AUC-ROC": 0.7226, "AUC-PR": 0.2712},        "val":   {"AUC-ROC": 0.6378, "AUC-PR": 0.2709},    },    "GBT tuned (maxIter=50, lr=0.05)": {        "train": {"AUC-ROC": 0.7375, "AUC-PR": 0.2901},        "test":  {"AUC-ROC": 0.7281, "AUC-PR": 0.2876},        "val":   {"AUC-ROC": 0.6749, "AUC-PR": 0.2984},    },}# MS4 results from the evaluation abovems4_model_label = f"GBT-PCA (k={K_MODEL}, 95% var)"rows = []for model_name, results in ms3_results.items():    for split in ["train", "test", "val"]:        r = results[split]        rows.append({            "Model": model_name, "Split": split,            "AUC-ROC": r["AUC-ROC"], "AUC-PR": r["AUC-PR"],        })for split in ["train", "test", "val"]:    r = gbt_pca_results[split]    rows.append({        "Model": ms4_model_label, "Split": split,        "AUC-ROC": round(r["AUC-ROC"], 4), "AUC-PR": round(r["AUC-PR"], 4),    })df_all = pd.DataFrame(rows)print("=== All Models Comparison ===")print(df_all.to_string(index=False))

In [ ]:
model_order = [    "RF (numTrees=50, maxDepth=8)",    "GBT default (maxIter=20, lr=0.1)",    "GBT tuned (maxIter=50, lr=0.05)",    ms4_model_label,]split_colors = {"train": "#4C72B0", "test": "#DD8452", "val": "#55A868"}fig, axes = plt.subplots(1, 2, figsize=(16, 6))x = np.arange(len(model_order))width = 0.25short_labels = ["RF\n(50 trees)", "GBT\n(default)", "GBT\n(tuned)", f"GBT\n(PCA-{K_MODEL})"]for ax_idx, metric in enumerate(["AUC-ROC", "AUC-PR"]):    ax = axes[ax_idx]    for i, (split, offset) in enumerate([("train", -width), ("test", 0), ("val", width)]):        vals = []        for model_name in model_order:            row = df_all[(df_all["Model"] == model_name) & (df_all["Split"] == split)]            vals.append(float(row[metric].values[0]) if len(row) > 0 else 0)        ax.bar(x + offset, vals, width, label=split, color=split_colors[split], edgecolor="white")    ax.set_title(f"{metric}: All Models")    ax.set_xticks(x); ax.set_xticklabels(short_labels, fontsize=9)    ax.set_ylabel(metric); ax.set_ylim(0.5, 1.0)    ax.legend(); ax.grid(axis="y", alpha=0.3)    ax.axvline(2.5, color="gray", linestyle="--", alpha=0.5, linewidth=1)    ax.text(2.55, 0.52, "MS4", fontsize=9, color="gray")plt.suptitle("Model Comparison: MS3 Baselines vs. MS4 GBT-PCA", fontsize=13)plt.tight_layout()plt.savefig(os.path.join(MS4_OUTPUT_DIR, "model_comparison.png"), dpi=150, bbox_inches="tight")plt.show()

In [ ]:
# Cold-start cohort breakdown for PCA model
print("=== GBT-PCA model: AUC-ROC by cold/warm cohort (test set) ===")
for cohort_val, cohort_name in [(0.0, "warm"), (1.0, "cold")]:
    preds_cohort = gbt_pca_fitted.transform(df_test_pca).filter(F.col("is_cold_article") == cohort_val)
    n = preds_cohort.count()
    if n > 100:
        auc = auc_evaluator.evaluate(preds_cohort)
        print(f"  {cohort_name} articles (n={n:,}): AUC-ROC = {auc:.4f}")
    else:
        print(f"  {cohort_name} articles (n={n}): too few to evaluate")

### 3.2 Fitting Analysis Discussion**Where does the GBT-PCA model fall on the fitting spectrum?**The train, test, and validation scores suggest the PCA-based GBT sits in roughly the same range as the stronger MS3 models. It does not look severely underfit, and it does not show signs of major overfitting either.- **Possible underfitting from compression**: if the 5% of variance dropped by PCA contains rare but useful label signal, such as infrequent one-hot category indicators, the PCA model can lose a small amount of performance compared with the full-feature tuned GBT.- **Why the drop should stay small**: keeping 95% of the variance should preserve most of the broad engagement signal, which is where the model gets much of its predictive power.- **Why overfitting should not get worse**: PCA removes correlation and redundancy in the inputs, so the trees are less likely to spend splits on multiple versions of the same signal.**How does dimensionality reduction affect the results?**Three practical effects matter here:1. **Most of the signal should stay intact** because the strongest predictors are high-variance engagement features that are likely to appear early in the PCA spectrum.2. **Cold-start behavior may improve slightly** if PCA removes noisy low-variance directions and forces the model to lean a bit more on the content features that cold articles still have.3. **Training can become cheaper** because the model is working with fewer input dimensions at each split.**Reasonable next improvements**- Try a larger `k`, such as the 99% variance threshold, to preserve more rare category signal.- Add dense text embeddings so cold articles carry more semantic information even before they accumulate engagement history.- Train separate cold and warm models, or use a gating step, since the cohort gap is persistent across experiments.- Retrain on rolling windows closer to the target period to reduce temporal distribution shift.

## Section 4: Update README.mdThe repository README was updated to serve as the main project report for the final submission. It now includes:- the project question and abstract- dataset scale and repository structure- links to the milestone notebooks- the preprocessing and modeling pipeline- model results, including the PCA-based second model- fitting analysis and cold-start discussion- conclusion sections for both models- the SDSC Expanse and Spark configuration used for the experimentsThat update satisfies the requirement to document the new work and provide links to the project notebooks in `README.md`.

## Section 5: Conclusion### 5.1 Model 1 Results (MS3, GBT on Full Features)Three distributed Spark MLlib models were trained on the full preprocessed feature set, which ends up at about 30 dimensions after one-hot encoding and scaling:| Model | Train AUC-ROC | Test AUC-ROC | Val AUC-ROC | Train to Test gap | Test to Val gap ||---|---|---|---|---|---|| RF (50 trees, depth 8) | 0.7123 | 0.7164 | 0.6538 | -0.0041 | -0.0626 || GBT default (20 rounds, lr=0.1) | 0.7200 | 0.7226 | 0.6378 | -0.0026 | -0.0848 || **GBT tuned (50 rounds, lr=0.05)** | **0.7375** | **0.7281** | **0.6749** | +0.0094 | -0.0532 |**GBT tuned is the strongest MS3 model.** Its most important feature is `rolling_popularity_24h` at 0.32 importance, which shows that recent article momentum is a stronger signal than any single content or user feature. The small train-to-test gap looks acceptable, while the drop from test to validation is better explained by time shift than by overfitting.Cold-start remains the main weakness. Most test rows involve cold articles, and those examples perform worse because the strongest engagement features are either missing or near zero for newly published content.**What could improve the MS3 models?**- Add contrastive or transformer-based article embeddings so cold articles have richer semantic features.- Use shorter rolling windows, such as 1 hour or 6 hours, to capture faster changes in momentum.- Build richer per-user preference features instead of relying mostly on history length.- Retrain on rolling windows to stay closer to the deployment period.---### 5.2 Model 2 Results (MS4, GBT on PCA-Reduced Features)PCA compresses the feature space into a smaller set of components while keeping 95% of the variance. The main takeaway is that the reduced space is still interpretable and still carries most of the useful predictive signal:- **PC1** is mostly an article engagement direction, driven by features such as `rolling_popularity_24h`, `log_total_inviews`, `log_total_pageviews`, and `total_read_time`.- **PC2** captures a mix of user engagement depth and session behavior, including `user_history_length`, `user_avg_read_time_hist`, `read_time`, and `scroll_percentage`.- Cold articles group toward the low-engagement side of the PCA projection, which supports the same cold-start story seen in the full-feature model.The GBT model trained on PCA features stays close to the tuned full-feature baseline, which suggests that the low-variance directions dropped by PCA do not carry much additional label signal in this setup.**What could improve the PCA plus GBT model?**- Increase `k` to keep more rare-category information.- Try supervised dimensionality reduction so the projection is optimized for class separation, not just variance.- Use sparse PCA if interpretability becomes more important.- Feed the reduced representation into a neural recommendation model as a next step.---### 5.3 Distributed Computing RoleBoth models rely heavily on distributed processing on SDSC Expanse, running on a 16 CPU, 128 GB node with PySpark `local[15]`. Spark is doing real work at every major stage:- exploding impressions into candidate rows at very large scale- downsampling negatives with window logic- computing rolling popularity and CTR features over time- fitting the preprocessing pipeline on millions of examples- computing PCA statistics before model training- training GBT models over large partitioned datasetsWithout Spark, this preprocessing and modeling pipeline would not fit comfortably into the project time budget on a standard single-machine workflow.

## Section 6: Predictions Analysis

We classify every test-set prediction into one of four buckets using a 0.5 decision threshold:

| Bucket | Condition | Meaning |
|---|---|---|
| TP | label=1, prediction=1 | Clicked article correctly ranked positive |
| TN | label=0, prediction=0 | Non-clicked article correctly ranked negative |
| FP | label=0, prediction=1 | Non-clicked article incorrectly promoted |
| FN | label=1, prediction=0 | Clicked article missed (demoted) |

We then break down FP and FN rates by cold vs. warm cohort to understand where the model fails.

In [ ]:
preds_test = (    gbt_pca_fitted.transform(df_test_pca)    .withColumn("prob_click", F.round(vector_to_array(F.col("probability"))[1], 6))    .withColumn(        "outcome",        F.when((F.col("label") == 1.0) & (F.col("prediction") == 1.0), F.lit("TP"))         .when((F.col("label") == 0.0) & (F.col("prediction") == 0.0), F.lit("TN"))         .when((F.col("label") == 0.0) & (F.col("prediction") == 1.0), F.lit("FP"))         .otherwise(F.lit("FN"))    )    .cache())outcome_counts = (    preds_test.groupBy("outcome").count().orderBy("outcome")    .toPandas().set_index("outcome"))total = outcome_counts["count"].sum()outcome_counts["pct"] = (outcome_counts["count"] / total * 100).round(2)print("=" * 50)print("CONFUSION MATRIX SUMMARY: Test Set (GBT-PCA)")print("=" * 50)print(outcome_counts.to_string())print(f"\nTotal test rows: {total:,}")TP = outcome_counts.loc["TP", "count"] if "TP" in outcome_counts.index else 0TN = outcome_counts.loc["TN", "count"] if "TN" in outcome_counts.index else 0FP = outcome_counts.loc["FP", "count"] if "FP" in outcome_counts.index else 0FN = outcome_counts.loc["FN", "count"] if "FN" in outcome_counts.index else 0precision = TP / (TP + FP) if (TP + FP) > 0 else 0recall    = TP / (TP + FN) if (TP + FN) > 0 else 0f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0print(f"\nPrecision : {precision:.4f}")print(f"Recall    : {recall:.4f}")print(f"F1        : {f1:.4f}")

In [ ]:
print("\n=== Sample predictions by outcome type ===")

for outcome_label in ["TP", "TN", "FP", "FN"]:
    print(f"\n--- {outcome_label} (5 examples) ---")
    preds_test.filter(F.col("outcome") == outcome_label) \
        .select(
            "impression_id", "candidate_article_id", "label",
            "prob_click", "prediction", "is_cold_article"
        ) \
        .limit(5).show(truncate=False)

In [ ]:
# FP/FN breakdown by cold vs. warm cohort
print("=== FP / FN rate by article cohort ===")
for cohort_val, cohort_name in [(0.0, "warm"), (1.0, "cold")]:
    sub = preds_test.filter(F.col("is_cold_article") == cohort_val)
    n_total = sub.count()
    n_fp    = sub.filter(F.col("outcome") == "FP").count()
    n_fn    = sub.filter(F.col("outcome") == "FN").count()
    n_pos   = sub.filter(F.col("label") == 1.0).count()
    n_neg   = sub.filter(F.col("label") == 0.0).count()
    fp_rate = n_fp / n_neg  if n_neg > 0 else 0
    fn_rate = n_fn / n_pos  if n_pos > 0 else 0
    print(f"\n  {cohort_name.upper()} articles (n={n_total:,}, pos={n_pos:,}, neg={n_neg:,})")
    print(f"    FP (false alarms)  : {n_fp:,}  ({100*fp_rate:.2f}% of negatives)")
    print(f"    FN (missed clicks) : {n_fn:,}  ({100*fn_rate:.2f}% of positives)")

In [ ]:
# Probability distribution by outcome typepreds_pd = (    preds_test    .select("prob_click", "outcome", "is_cold_article")    .sample(fraction=min(1.0, 200_000 / preds_test.count()), seed=42)    .toPandas())fig, axes = plt.subplots(1, 2, figsize=(15, 5))# Panel 1: prob distribution by outcomeax = axes[0]colors_map = {"TP": "#55A868", "TN": "#4C72B0", "FP": "#C44E52", "FN": "#DD8452"}for outcome_label, color in colors_map.items():    sub = preds_pd[preds_pd["outcome"] == outcome_label]["prob_click"]    if len(sub) > 10:        ax.hist(sub, bins=50, alpha=0.55, color=color, label=f"{outcome_label} (n={len(sub):,})", density=True)ax.axvline(0.5, color="black", linestyle="--", linewidth=1.5, label="threshold=0.5")ax.set_title("Click Probability Distribution by Outcome (Test Set)")ax.set_xlabel("P(click)")ax.set_ylabel("Density")ax.legend()ax.grid(alpha=0.3)# Panel 2: FP and FN by cold/warmax = axes[1]cohorts = ["warm", "cold"]fp_rates_plot, fn_rates_plot = [], []for cohort_val, cohort_name in [(0.0, "warm"), (1.0, "cold")]:    sub = preds_pd[preds_pd["is_cold_article"] == cohort_val]    n_neg = (sub["outcome"].isin(["TN", "FP"])).sum()    n_pos = (sub["outcome"].isin(["TP", "FN"])).sum()    fp_rates_plot.append(100 * (sub["outcome"] == "FP").sum() / n_neg if n_neg > 0 else 0)    fn_rates_plot.append(100 * (sub["outcome"] == "FN").sum() / n_pos if n_pos > 0 else 0)x = np.arange(len(cohorts))width = 0.35ax.bar(x - width/2, fp_rates_plot, width, label="FP rate (% of negatives)", color="#C44E52", alpha=0.8)ax.bar(x + width/2, fn_rates_plot, width, label="FN rate (% of positives)", color="#DD8452", alpha=0.8)ax.set_title("FP and FN Rates by Article Cohort")ax.set_xticks(x)ax.set_xticklabels(["Warm Articles", "Cold Articles"])ax.set_ylabel("Error Rate (%)")ax.legend()ax.grid(axis="y", alpha=0.3)plt.suptitle("GBT-PCA Model: Test Set Prediction Analysis", fontsize=13)plt.tight_layout()plt.savefig(os.path.join(MS4_OUTPUT_DIR, "prediction_analysis.png"), dpi=150, bbox_inches="tight")plt.show()

## Cleanup

In [ ]:
for df in [df_train_sampled, df_val_sampled,
           df_train_prep, df_test_prep, df_val_prep,
           df_train_pca, df_test_pca, df_val_pca,
           preds_test]:
    try:
        df.unpersist()
    except Exception:
        pass

spark.stop()
print("Spark session stopped.")